# Building goSPL inputs for a global continental-flux model

This notebook prepares the input files required to run a **global** `gospl` landscape evolution simulation that tracks sediment flux from continents to oceans. Starting from regular (lon/lat) paleo-reconstruction grids, it (1) generates an unstructured spherical **Voronoi/triangular mesh** with [JIGSAW](https://github.com/dengwirda/jigsaw) via UXarray, (2) interpolates the forcing fields (elevation, plate velocities, rainfall) onto that mesh, (3) derives the **vertical tectonic forcing** from successive paleo-elevation maps, and (4) writes the compressed `.npz` mesh and forcing files that `gospl` reads at run time. A second part shows how to build a **depth-dependent resolution** mesh (fine on continents, coarse in the deep ocean).

By the end you will have an `init.vtk` you can inspect in ParaView and the `mesh*.npz` / `forcing*.npz` files needed to launch a simulation.

Import required Python packages for this notebook.


In [ ]:
import importlib, subprocess, sys

def ensure_installed(package):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

ensure_installed("pysheds")

Import required Python packages for this notebook.


In [ ]:
import os
import shutil
import numpy as np
import xarray as xr

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from scripts import umeshFcts as ufcts

We pick a uniform target resolution and build the mesh. The table above maps `widthCell` to the resulting edge lengths and node count.

| Parameter | Symbol | Value | Physical meaning |
| --------- | ------ | ----- | ---------------- |
| `widthCell` | $\Delta x$ | 30 | Target Voronoi cell width in km (here ~660k nodes, mean edge ~17 km) |
| `input_path` | — | `input_30` | Output folder collecting the generated mesh files |

`buildGlobalMeshSimple` runs JIGSAW to produce a globally uniform spherical mesh; if the mesh already exists it is not regenerated.

# Create a global mesh for goSPL

Create an unstructured grid for a given cell width. The method relies on the UXarray and jigsaw libraries.

**In case where the mesh already exists it will not be recreated.**

Spherical mesh resolution km

| cell_width | edge_min  | edge_max | edge_mean | nodeNb |
| ---------- | ----------  | ---------- | ---------- | ---------- |
| 5 | 1.1 | 4.5 | 2.8 | 23632811 |
| 8 |  1.8 | 7.2 | 4.6  | 9236387 | 
| 10 | 2.2 | 8.9 | 5.7 | 5912778 |
| 15 | 3.3 | 13.1 | 8.6 | 2629742 |
| 20 | 4.5 | 18 | 11.5 | 1480168 |
| 25 | 5.6 | 22.4 | 14.4 | 947701 |
| 30 | 6.8 | 26.4 | 17.2 | 658525 |
| 35 | 8 | 30.5 | 20.1 |  484009 |

In [ ]:
widthCell = 30
input_path = "input_"+str(widthCell) 

# Build the mesh
ufcts.buildGlobalMeshSimple(widthCell, input_path)

Run shell commands for file or mesh management.


In [ ]:
!mv mesh.jig mesh.log mesh.msh mesh-MESH.msh mesh-HFUN.msh mesh_triangles.nc cellWidthVsLatLon.nc $input_path

## Map variables on the UGRID 

We will now map global variables on this unstructured grid. In goSPL, typical variables would be:

- elevation (in m)
- vertical and horizontal tectonic forcing (displacement rates in m/yr)
- precipitation (in m/yr)
- dynamic topography (in m/yr)

Usually they will be provided in the form of `netcdf` or `geotiff` files. In both cases, the `xarray` or `rioxarray` libraries will allow you to open those files conveniently.

> Here we will use a netcdf grid containing all of these variables (except dynamic topography) for a give time interval.

In [ ]:
# Loading the nc regular file
ncgrid = xr.open_dataset('data/250.nc')
ncgrid

In case the file contains more variables than the ones you need for goSPL, you can select only the necessary ones:

In [ ]:
ncgrid = ncgrid[['h','vx','vy','vz','rain']]
ncgrid

Loading the UGRID file


### Reading the mesh geometry

We extract the Cartesian node coordinates (`xCell`, `yCell`, `zCell` on the unit sphere scaled to Earth radius) and the face connectivity (`cellsOnVertex`, converted to 0-based indexing for the triangular dual mesh). The `dcEdge` field gives the spacing between neighbouring cell centres in metres, reported here as the actual min/max/mean edge lengths so you can confirm the realised resolution matches the requested `widthCell`.

In [ ]:
# Loading the UGRID file
ufile = input_path+'/mesh_'+str(widthCell)+'km.nc'
mapds = xr.open_dataset(ufile) 

# Perform the interpolation (bilinear) 
var_path = 'vars_'+str(widthCell)
var_name = 'step_250'
if os.path.exists(var_path):
    shutil.rmtree(var_path)
ufcts.inter2UGRID(ncgrid,mapds,var_path,var_name,type='face')
data_ds = xr.open_dataset(var_path + '/' + var_name + '.nc')

In the `var_path` folder, you will find interpolated variables for the the UGRID (one file per variable) 

In [ ]:
# Extract nodes and faces information
n_nodes = mapds.dims['nCells']
ucoords = np.zeros((n_nodes, 3))
ucoords[:, 0] = mapds['xCell'].values
ucoords[:, 1] = mapds['yCell'].values
ucoords[:, 2] = mapds['zCell'].values
ufaces = mapds['cellsOnVertex'].values - 1 
print(f"Number of nodes: {len(ucoords)} | Number of faces: {len(ufaces)}")

# Get information about your mesh:
dcEdge = mapds['dcEdge'].values  # in metres
edge_min = np.round(dcEdge.min() /1000.+0.,2)
edge_max = np.round(dcEdge.max() /1000.+0.,2)
edge_mean = np.round(dcEdge.mean() /1000.+0.,2)
print("edge range (km): min ",edge_min," | max ",edge_max," | mean ",edge_mean)

Save voronoi mesh for visualisation purposes


In [ ]:
# Save voronoi mesh for visualisation purposes
saveVoro = True

if saveVoro:
    from mpas_tools.viz.paraview_extractor import extract_vtk
    extract_vtk(
            filename_pattern=ufile,
            variable_list='areaCell',
            dimension_list=['maxEdges=','nVertLevels=', 'nParticles='], 
            mesh_filename=ufile,
            out_dir=input_path, 
            ignore_time=True,
            # lonlat=True,
            xtime='none'
        )
    print("You could now visualise in Paraview (wireframe) the produced voronoi mesh!")
    print("This is a vtp mesh called: ", input_path+'/staticFieldsOnCells.vtp')

> You might want to check that everything went according to plan and look at the mesh and variables that will be used in goSPL.

To do so, we will build a `vtk` file that could be visualised in Paraview...

In [ ]:
checkMesh = True

if checkMesh:
    import meshio

    paleovtk = input_path+"/init.vtk"

    vlist = list(data_ds.keys())
    vdata = []
    for k in vlist:
        vdata.append(data_ds[k].values)

    list_data = dict.fromkeys(el for el in vlist)
    list_data.update((k, vdata[i]) for i, k in enumerate(list_data))

    # Define mesh
    vis_mesh = meshio.Mesh(ucoords, {"triangle": ufaces}, 
                           point_data = list_data,
                        )
    # Write it disk
    meshio.write(paleovtk, vis_mesh)
    print("Writing VTK input file as {}".format(paleovtk))

## Extracting vertical tectonic forcing from successive paleo-elevation maps

In case where you want to run a model with horizontal displacement rates then you could choose to adjust the tectonic forcing based on next time step paleo-elevation. 

To illustrate how this could be done we will use the `netcdf` grids provided in the data folder (*i.e.* 250.nc and 251.nc).

In [ ]:
# Loading the nc files
ncgrid_250 = xr.open_dataset('data/250.nc')
ncgrid_251 = xr.open_dataset('data/251.nc')

# Store next elevation as a new variable in the grid to interpolate
ncgrid_250 = ncgrid_250[['h','vx','vy','vz','rain']]
ncgrid_250['next_h'] = ncgrid_251['h']
ncgrid_250

`get_Tectonic` reconstructs the **vertical tectonic uplift/subsidence rate** `tec` (m/yr) by comparing the current and next-step paleo-elevations after accounting for the horizontal advection implied by the plate velocities, using inverse-distance weighting (`mthd='IDW'`) to back-track each node.

| Parameter | Symbol | Value | Physical meaning |
| --------- | ------ | ----- | ---------------- |
| `dt` | $\Delta t$ | 1.e6 | Time interval between the two paleo-maps, in years (1 Myr) |
| `zkeys` | $z, z_{t+1}$ | `['h','next_h']` | Current and next-step elevation fields (m) used to infer net vertical motion |
| `vkeys` | $v_x,v_y,v_z$ | `['vx','vy','vz']` | Horizontal/vertical plate displacement-rate components (m/yr) |
| `dkey` | — | `None` | Optional dynamic-topography rate; if set, `tec` combines tectonic + dynamic topography |
| `mthd` | — | `'IDW'` | Interpolation method for the velocity back-tracking |

Similar to what was done before we interpolate the structured variables to the UGRID mesh:

In [ ]:
# Loading the UGRID file
ufile = input_path+'/mesh_'+str(widthCell)+'km.nc'
mapds = xr.open_dataset(ufile) 

# Perform the interpolation (bilinear) 
var_path = 'vars_'+str(widthCell)
var_name = 'step_up_250'
ufcts.inter2UGRID(ncgrid_250,mapds,var_path,var_name,type='face')

We then extract the tectonic forcing based on the displacement rates:

The forcing file bundles the per-node fields `gospl` ingests each time step:

| Key | Field | Units | Physical meaning |
| --- | ----- | ----- | ---------------- |
| `vxyz` | $\mathbf{v}=(v_x,v_y,v_z)$ | m/yr | Plate displacement-rate vector (horizontal advection of the surface) |
| `t` | `tec` | m/yr | Vertical tectonic forcing (uplift positive, subsidence negative) |
| `r` | rain | m/yr | Precipitation rate driving runoff and the stream-power erosion $E = K A^m S^n$ |
| `nz` | `next_h` | m | Target next-step paleo-elevation, used by `gospl` when `zfit` constrains regions to a known elevation |

In [ ]:
data_file = [var_path+'/'+var_name+'.nc']

dt = 1.e6
zkeys = ['h','next_h']
vkeys = ['vx','vy','vz']
# If you have a dynamic topography variable you can specify its corresponding key here for example dkey = ['dynt'] 
dkey = None

data_ds = ufcts.get_Tectonic(ufile, *data_file, vkeys, zkeys, dkey, dt, mthd='IDW')

| Parameter | Symbol | Value | Physical meaning |
| --------- | ------ | ----- | ---------------- |
| `widthCell` | $\Delta x_{bg}$ | 50 | Background (coarse) cell width in km used to bootstrap the mesh |
| `highres` | $\Delta x_{min}$ | 25 | Fine cell width in km applied over continental/shallow regions |
| `lowres` | $\Delta x_{max}$ | 80 | Coarse cell width in km applied over the deep ocean |
| `reso_contour` | $z_c$ | -50 | Bathymetric contour (m) separating fine (above) from coarse (below) refinement |
| `input_path` | — | `input_25_80` | Output folder for this variable-resolution mesh |

> In case where you have specified a dynamic topography variable, your tectonic variable `tec` is a combination of the tectonic and dynamic topography components.

## goSPL input generation

We will now create the inputs for goSPL. We first start by creating the input mesh defining our UGRID structure:

In [ ]:
meshname = var_path+"/mesh"
np.savez_compressed(meshname, v=ucoords, c=ufaces, 
                    z=data_ds.h.data
                    )

Now we save the forcing conditions (displacement rates, tectonic, precipitation...). Here you have the option to also add the next time step elevation, this will then be used in goSPL to force the model to match with the next paleo-elevation for specific regions (by defining the `zfit` parameter in the input file).

In [ ]:
forcname = var_path+"/forcing250"

vel = np.zeros(ucoords.shape)
vel[:,0] = data_ds.vx.data
vel[:,1] = data_ds.vy.data
vel[:,2] = data_ds.vz.data
np.savez_compressed(forcname, 
                    vxyz=vel, 
                    t=data_ds.tec.data, 
                    r=data_ds.rain.data,
                    nz=data_ds.next_h.data,
                    )


# Depth-dependent mesh resolution

We show how a similar approach could be used to build a mesh with variable grid resolution. 


Here as an example, we will use a coarser background mesh (cell width set to 50 km) for the ocean region (<-50 m depth) and a finer mesh for continental region (cell width set to 20 km).

### Building the variable-width field

From the coarse mesh we compute, for every point, its **distance to the -50 m coastline contour** (`distanceCoasts`). `cellWidthVsLatLonFuncDist` then turns that distance into a target cell width that ramps from `highres` (25 km) near/landward of the contour up to `lowres` (80 km) once a node is more than `maxdist = 2.5e6` m (2500 km) into the deep ocean. The result is a global lon/lat array of desired resolutions.

In [ ]:
widthCell = 50
highres = 25
lowres = 80
reso_contour = -50
input_path = "input_"+str(highres)+"_"+str(lowres)

# Build a background mesh with a 50 km width cell
ufcts.buildGlobalMeshSimple(widthCell, input_path)

### Visualising the target resolution field

The map below shows the requested cell width (km) on a PlateCarree projection: continents and shelves (above -50 m, within ~2500 km of the coast) appear at the fine 25 km resolution, grading to 80 km in the open ocean. Use it as a sanity check that the refinement follows the coastlines before feeding `cellWidth` to JIGSAW.

Run shell commands for file or mesh management.


In [ ]:
!mv mesh.jig mesh.log mesh.msh mesh-MESH.msh mesh-HFUN.msh mesh_triangles.nc CellWidthVsLatLon.nc $input_path

We then perform the same steps as before to interpolate the variables from the input onto the UGRID:

In [ ]:
# Loading the nc regular file
ncgrid = xr.open_dataset('data/250.nc')[['h','rain']]

# Loading the UGRID file
ufile = input_path+'/mesh_'+str(widthCell)+'km.nc'
mapds = xr.open_dataset(ufile) 

# Perform the interpolation (bilinear) 
var_path = 'vars_'+str(highres)+"_"+str(lowres)
var_name = 'coarse_250'
if os.path.exists(var_path):
    shutil.rmtree(var_path)
ufcts.inter2UGRID(ncgrid,mapds,var_path,var_name,type='face')
data_ds = xr.open_dataset(var_path + '/' + var_name + '.nc')

# Extract nodes and faces information
n_nodes = mapds.dims['nCells']
ucoords = np.zeros((n_nodes, 3))
ucoords[:, 0] = mapds['xCell'].values
ucoords[:, 1] = mapds['yCell'].values
ucoords[:, 2] = mapds['zCell'].values
ufaces = mapds['cellsOnVertex'].values - 1 
print(f"Number of nodes: {len(ucoords)} | Number of faces: {len(ufaces)}")

# Get information about your mesh:
dcEdge = mapds['dcEdge'].values  # in metres
edge_min = np.round(dcEdge.min() /1000.+0.,2)
edge_max = np.round(dcEdge.max() /1000.+0.,2)
edge_mean = np.round(dcEdge.mean() /1000.+0.,2)
print("edge range (km): min ",edge_min," | max ",edge_max," | mean ",edge_mean)

We will now specify the bathymetry threshold to generate our refinement grid, to do so we buid a vtk Mesh to extract the distances to the chosen contour (here -50 m):

In [ ]:
vtkMesh = ufcts.generateVTKmesh(ucoords, ufaces)
dcoast = ufcts.distanceCoasts(vtkMesh, ucoords, data_ds.h.values, reso_contour)
ngrd = ufcts.getGridCoast(ncgrid, mapds, dcoast, input_path)
ds, cellWidth = ufcts.cellWidthVsLatLonFuncDist(ngrd, width=[highres,lowres], maxdist=2.5e6)

Run the following data preparation or mesh generation step.


In [ ]:
plotWeight = True
if plotWeight:
    import cartopy.crs as ccrs
    import matplotlib.pyplot as plt
    fig = plt.figure(figsize=[8.0, 5.0])
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_global()
    im = ax.imshow(cellWidth, origin='lower',
                   vmin=cellWidth.min()-5,
                   vmax=cellWidth.max()+5,
                    transform=ccrs.PlateCarree(),
                    extent=[-180, 180, -90, 90], cmap='RdYlBu',
                    zorder=0)
    plt.title(
        'Grid cell size, km, min: {:.1f} max: {:.1f}'.format(
            cellWidth.min(),cellWidth.max()),fontsize=10)
    plt.colorbar(im, shrink=.60)
    fig.canvas.draw()
    plt.tight_layout()
    # plt.savefig(input_path+'/cellWidthGlobal.png', bbox_inches='tight')
    plt.show()
    plt.close()

We will now use this variable width array to generate our mesh.

As for the uniform-resolution case, we write the two `.npz` files `gospl` reads: `mesh.npz` holding the node coordinates `v`, the triangular connectivity `c` and the initial elevation `z` (m), and `rain250.npz` holding the precipitation field `r` (m/yr). These, together with a `gospl` YAML input file pointing at them, are all that is needed to launch the simulation on the refined mesh.

In [ ]:
# Build the mesh
lon = ncgrid.lon.values
lat = ncgrid.lat.values
ufcts.refineGlobalMesh(cellWidth, lon, lat, input_path)

Run shell commands for file or mesh management.


In [ ]:
!mv mesh.jig mesh.log mesh.msh mesh-MESH.msh mesh-HFUN.msh mesh_triangles.nc CellWidthVsLatLon.nc $input_path

This will generate a new mesh called `mesh_refine.nc` over which we are going to interpolate our goSPL variable (elevation and rainfall). 

In [ ]:
# Loading the UGRID file
ufile = input_path+'/mesh_refine.nc'
mapds = xr.open_dataset(ufile) 

# Perform the interpolation (bilinear) 
var_path = 'vars_'+str(highres)+"_"+str(lowres)
var_name = 'refine_250'
ufcts.inter2UGRID(ncgrid_250,mapds,var_path,var_name,type='face')
data_ds = xr.open_dataset(var_path + '/' + var_name + '.nc')

# Extract nodes and faces information
n_nodes = mapds.dims['nCells']
ucoords = np.zeros((n_nodes, 3))
ucoords[:, 0] = mapds['xCell'].values
ucoords[:, 1] = mapds['yCell'].values
ucoords[:, 2] = mapds['zCell'].values
ufaces = mapds['cellsOnVertex'].values - 1 
print(f"Number of nodes: {len(ucoords)} | Number of faces: {len(ufaces)}")

# Get information about your mesh:
dcEdge = mapds['dcEdge'].values  # in metres
edge_min = np.round(dcEdge.min() /1000.+0.,2)
edge_max = np.round(dcEdge.max() /1000.+0.,2)
edge_mean = np.round(dcEdge.mean() /1000.+0.,2)
print("edge range (km): min ",edge_min," | max ",edge_max," | mean ",edge_mean)

Save voronoi mesh for visualisation purposes


In [ ]:
# Save voronoi mesh for visualisation purposes
saveVoro = True

if saveVoro:
    from mpas_tools.viz.paraview_extractor import extract_vtk
    extract_vtk(
            filename_pattern=ufile,
            variable_list='areaCell',
            dimension_list=['maxEdges=','nVertLevels=', 'nParticles='], 
            mesh_filename=ufile,
            out_dir=input_path, 
            ignore_time=True,
            # lonlat=True,
            xtime='none'
        )
    print("You could now visualise in Paraview (wireframe) the produced voronoi mesh!")
    print("This is a vtp mesh called: ", input_path+'/staticFieldsOnCells.vtp')

## goSPL mesh input

In [ ]:
meshname = var_path+"/mesh"
np.savez_compressed(meshname, v=ucoords, c=ufaces, 
                    z=data_ds.h.data
                    )

forcname = var_path+"/rain250"
np.savez_compressed(forcname, 
                    r=data_ds.rain.data,
                    )